# Week 4: LLM and ML Benchmarking

This notebook shows a simple way to test prompt versions across three LLM endpoints.

Goal: choose robust prompts and document failure modes for each model.

## Step 1: Imports

What this does: imports libraries for data processing, prompt parsing, and model calls.

Why it matters: this keeps each later step focused on logic, not setup.

Principle: set up your toolbox first so experiments are repeatable and easy to debug.

In [14]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

openai = __import__('openai')
OpenAI = openai.OpenAI

MODEL_ENDPOINTS = [
    {'label': 'qwen2.5-7b', 'model_name': 'Qwen2.5-7B', 'host': 'localhost', 'port': 8000},
    {'label': 'qwen2.5-7b-instruct', 'model_name': 'Qwen2.5-7B-Instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'qwen2.5-coder-7b-instruct', 'model_name': 'Qwen2.5-Coder-7B-Instruct', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'

# ── Paths ──────────────────────────────────────────────────────────
ROOT = Path('.').resolve()
DATA_CSV = ROOT/'week3_artifacts/week2_data.csv'
ML_CSV = ROOT/'week3_artifacts/ml_results.csv'
SPLIT_CSV = ROOT/'week3_artifacts/splits_indexed.csv'
PROMPT_DIR = ROOT / "Prompts"
PROMPT_DIR.mkdir(exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────
ID_COL = "timestamp"
SPLIT_COL = "split"
TARGET_REF_COL = "delta_T_above_T0_TN"
HORIZONS = [240, 1440]
TARGET_COLS = ["target_240", "target_1440"]
ML_ID_COL = "Test_Idx"
ML_COLS = ["Multi-XGB_+240m","Multi-XGB_+1440m"]

BASE_FEATURES = [
    "cumulative_heat_input", "elapsed_injection_min", "net_flow_rolling_6h",
    "TC_INT_delta", "T_gradient_INT_TC", "days_since_injection",
    "hour_sin", "hour_cos", "delta_T_above_T0_TN",
]

FEATURES_FULL = [
    "cumulative_heat_input", "elapsed_injection_min", "net_flow_rolling_6h",
    "TC_INT_delta", "T_gradient_INT_TC", "days_since_injection",
    "hour_sin", "hour_cos", "delta_T_above_T0_TN",
    "target_lag_15", "flow_lag_15", "target_lag_30", "flow_lag_30",
    "target_lag_60", "flow_lag_60",
]

RESPONSE_SCHEMA = '{"target_240": <float>, "target_1440": <float>, "confidence": <0-1>}'

print(f"ROOT:       {ROOT}")
print(f"DATA_CSV:   {DATA_CSV}  (exists={DATA_CSV.exists()})")
print(f"PROMPT_DIR: {PROMPT_DIR}")
print(f"Features:   {len(FEATURES_FULL)}")



ROOT:       /home/jupyter-xue.li/ByteMe_week_4
DATA_CSV:   /home/jupyter-xue.li/ByteMe_week_4/week3_artifacts/week2_data.csv  (exists=True)
PROMPT_DIR: /home/jupyter-xue.li/ByteMe_week_4/Prompts
Features:   15


## Step 2: Load Week 3 test data

What this does: loads the Week 3 held-out split and defines the allowed label set.

Why it matters: prompt comparison is only meaningful when all versions are tested on the same target and data.

Principle: control your dataset before comparing methods, otherwise comparisons are not fair.

In [3]:
# Load only the columns we need
df = pd.read_csv(DATA_CSV)[[ID_COL]+BASE_FEATURES].copy()
df_ml = pd.read_csv(ML_CSV,index_col=ML_ID_COL)[ML_COLS].copy()
df_split = pd.read_csv(SPLIT_CSV).copy()

# Lag features
for lag in [15, 30, 60]:
    df[f"target_lag_{lag}"] = df[TARGET_REF_COL].shift(lag)
    df[f"flow_lag_{lag}"] = df["net_flow_rolling_6h"].shift(lag)

# Forward targets
for h in HORIZONS:
    df[f"target_{h}"] = df[TARGET_REF_COL].shift(-h)

# Split
idx_train = df_split[SPLIT_COL] == "train"
idx_test = df_split[SPLIT_COL] == "test"
df_train = df.loc[idx_train].copy()
df_test = df.loc[idx_test].copy()

# Append ML predictions
df_test = pd.merge(df_test, df_ml,left_index=True, right_index=True)

print(f"Train rows: {len(df_train):,}   Test rows: {len(df_test):,}")
display(df_test.head())

Train rows: 71,924   Test rows: 24,235
Columns:    ['timestamp', 'cumulative_heat_input', 'elapsed_injection_min', 'net_flow_rolling_6h', 'TC_INT_delta', 'T_gradient_INT_TC', 'days_since_injection', 'hour_sin', 'hour_cos', 'delta_T_above_T0_TN', 'target_lag_15', 'flow_lag_15', 'target_lag_30', 'flow_lag_30', 'target_lag_60', 'flow_lag_60', 'target_240', 'target_1440']


,timestamp,cumulative_heat_input,elapsed_injection_min,net_flow_rolling_6h,TC_INT_delta,T_gradient_INT_TC,days_since_injection,hour_sin,hour_cos,delta_T_above_T0_TN,target_lag_15,flow_lag_15,target_lag_30,flow_lag_30,target_lag_60,flow_lag_60,target_240,target_1440,Multi-XGB_+240m,Multi-XGB_+1440m
72001,2025-02-01 20:01:00,4.238254e+08,72001.0,1.485943,0.768405,-5.345621,50.000694,-0.863836,0.503774,5.607457,5.570220,1.485913,5.606272,1.485947,5.584420,1.485944,5.606484,5.576771,5.606494,5.635816
72002,2025-02-01 20:02:00,4.238291e+08,72002.0,1.485940,0.151337,-5.331847,50.001389,-0.861629,0.507538,5.598087,5.604220,1.485904,5.568302,1.485941,5.592866,1.485954,5.572344,5.554104,5.577637,5.612080
72003,2025-02-01 20:03:00,4.238329e+08,72003.0,1.485936,-0.175045,-5.325941,50.002083,-0.859406,0.511293,5.568759,5.599118,1.485908,5.606115,1.485946,5.612416,1.485928,5.600141,5.569208,5.586841,5.620599
72004,2025-02-01 20:04:00,4.238366e+08,72004.0,1.485961,-0.910542,-5.348824,50.002778,-0.857167,0.515038,5.605122,5.571350,1.485932,5.604886,1.485953,5.577321,1.485904,5.607526,5.580339,5.606325,5.635647
72005,2025-02-01 20:05:00,4.238404e+08,72005.0,1.485971,0.228062,-5.353266,50.003472,-0.854912,0.518773,5.582435,5.609779,1.485940,5.573950,1.485927,5.607153,1.485920,5.563109,5.552715,5.567493,5.608487


,Multi-XGB_+240m,Multi-XGB_+1440m
Test_Idx,,
71636,5.613284,5.636293
71637,5.603914,5.626924
71638,5.574586,5.609267
71639,5.610949,5.633959
71640,5.588262,5.616392


## Step 3: Define prompt versions

What this does: creates alternative prompt templates (baseline and few-shot) and one parser for all outputs.

Why it matters: you can test formatting and accuracy differences while keeping evaluation logic constant.

Principle: change one variable at a time. Here, prompt text changes while parser and scoring stay fixed.

In [4]:
# ═══════════════════════════════════════════════════════════════════
# Sampling
# ═══════════════════════════════════════════════════════════════════
def sample_evenly_spaced(df, num_examples, features=None):
    """Sample evenly-spaced rows across time, dropping rows with NaN."""
    cols = list(features or [c for c in df.columns if c != SPLIT_COL])
    cols_needed = list(set(cols) | set(TARGET_COLS))
    df_clean = df[[c for c in cols_needed if c in df.columns]].dropna()
    if len(df_clean) < num_examples:
        raise ValueError(f"Only {len(df_clean)} complete rows, need {num_examples}")
    idxs = np.linspace(0, len(df_clean) - 1, num_examples, dtype=int)
    return df_clean.iloc[idxs]


# ═══════════════════════════════════════════════════════════════════
# Formatters
# ═══════════════════════════════════════════════════════════════════
def format_verbose(row, features, targets=TARGET_COLS, precision=4):
    """Named key=value pairs in natural language."""
    feat_str = ", ".join(f"{c}={row[c]:.{precision}f}" for c in features)
    tgt = {c: round(row[c], precision) for c in targets}
    tgt["confidence"] = 1.0
    prompt = f"Given features: {feat_str}\nPredict temperature delta and confidence."
    completion = json.dumps(tgt)
    return {"prompt": prompt, "completion": completion}


def format_compact(row, features, targets=TARGET_COLS, precision=2):
    """Compact input array, RESPONSE_SCHEMA output."""
    inp = [round(row[c], precision) for c in features]
    tgt = {c: round(row[c], precision) for c in targets}
    tgt["confidence"] = 1.0
    return {"i": inp, "completion": json.dumps(tgt)}


FORMATTERS = {"verbose": format_verbose, "compact": format_compact}


# ═══════════════════════════════════════════════════════════════════
# System message
# ═══════════════════════════════════════════════════════════════════
def build_system_message(style, features=FEATURES_FULL):
    """Build the system instruction for the LLM."""
    msg = (
        "You are a geothermal temperature forecasting model.\n"
        f"Respond ONLY with valid JSON matching this schema:\n{RESPONSE_SCHEMA}\n"
        "target_240 = predicted delta_T at 4 h, target_1440 = at 24 h.\n"
        "confidence = your estimated probability that the prediction is "
        "within 10% of the true value (0-1)."
    )
    if style == "compact":
        msg += f"\nFeature order: {features}\nOutput order: {TARGET_COLS + ['confidence']}"
    return msg


# ═══════════════════════════════════════════════════════════════════
# Zero-shot prompt builder
# ═══════════════════════════════════════════════════════════════════
def build_zero_shot_prompt(test_row, style="verbose", features=FEATURES_FULL, precision=None):
    """Build a zero-shot prompt (system + single query, no examples)."""
    prec = precision or (4 if style == "verbose" else 2)
    fmt_fn = FORMATTERS[style]
    system = build_system_message(style, features)

    test_ex = fmt_fn(test_row, features, TARGET_COLS, prec)
    if style == "verbose":
        query = test_ex["prompt"]
    else:
        query = json.dumps(test_ex["i"])

    return {"system": system, "user": f"Q: {query}\nA:", "parse_hint": "json"}


# ═══════════════════════════════════════════════════════════════════
# Few-shot generation & prompt builder
# ═══════════════════════════════════════════════════════════════════
def generate_few_shot(df_train, num_examples=5, features=FEATURES_FULL,
                      style="verbose", precision=None):
    """Sample training examples and format them as few-shot examples."""
    fmt_fn = FORMATTERS[style]
    prec = precision or (4 if style == "verbose" else 2)
    samples = sample_evenly_spaced(df_train, num_examples, features)
    examples = [fmt_fn(row, features, TARGET_COLS, prec) for _, row in samples.iterrows()]
    return {
        "meta": {
            "style": style,
            "features": features,
            "targets": TARGET_COLS,
            "response_schema": RESPONSE_SCHEMA,
            "num_examples": num_examples,
        },
        "examples": examples,
    }


def build_few_shot_prompt(few_shot, test_row, features=None, style=None, precision=None):
    """Assemble system + N example Q/A pairs + test query."""
    meta = few_shot["meta"]
    feat_list = features or meta["features"]
    st = style or meta["style"]
    prec = precision or (4 if st == "verbose" else 2)
    fmt_fn = FORMATTERS[st]

    system = build_system_message(st, feat_list)

    # Few-shot examples block
    lines = []
    for ex in few_shot["examples"]:
        if st == "verbose":
            lines.append(f"Q: {ex['prompt']}\nA: {ex['completion']}")
        else:
            lines.append(f"Q: {json.dumps(ex['i'])}\nA: {ex['completion']}")

    # Test query
    test_ex = fmt_fn(test_row, feat_list, TARGET_COLS, prec)
    if st == "verbose":
        query = test_ex["prompt"]
    else:
        query = json.dumps(test_ex["i"])

    user = "\n\n".join(lines) + f"\n\nQ: {query}\nA:"
    return {"system": system, "user": user, "parse_hint": "json"}


# ═══════════════════════════════════════════════════════════════════
# Save / load helpers
# ═══════════════════════════════════════════════════════════════════
def save_prompts(data, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    print(f"Saved → {output_path}  ({output_path.stat().st_size / 1024:.1f} KB)")
    
# ═══════════════════════════════════════════════════════════════════
# Parse LLM response
# ═══════════════════════════════════════════════════════════════════
def parse_response(raw_text, valid_range=None):
    """Extract predictions + confidence from LLM response string.

    Args:
        raw_text:     Raw LLM output string.
        valid_range:  Optional (min, max) tuple; predictions outside this range
                      are flagged as invalid.

    Returns:
        (pred_dict, confidence, is_valid, error_reason)
        pred_dict:    {'target_240': float, 'target_1440': float} or None
        confidence:   float in [0, 1] or np.nan
        is_valid:     bool
        error_reason: None when valid, else one of
                      'empty_response', 'invalid_json', 'missing_targets',
                      'non_numeric', 'out_of_range'
    """
    if not isinstance(raw_text, str) or not raw_text.strip():
        return None, np.nan, False, 'empty_response'

    text = raw_text.strip()

    # --- Try to extract JSON object or array from the response ---
    m = re.search(r'\{.*\}|\[.*\]', text, flags=re.DOTALL)
    candidate = m.group(0) if m else text

    try:
        payload = json.loads(candidate)
    except Exception:
        return None, np.nan, False, 'invalid_json'

    # --- Normalise compact array [t240, t1440, conf] → dict ---
    if isinstance(payload, list):
        if len(payload) < 2:
            return None, np.nan, False, 'missing_targets'
        payload = {
            'target_240':  payload[0],
            'target_1440': payload[1],
            'confidence':  payload[2] if len(payload) > 2 else np.nan,
        }

    # --- Extract targets ---
    t240  = payload.get('target_240')
    t1440 = payload.get('target_1440')
    if t240 is None or t1440 is None:
        return None, np.nan, False, 'missing_targets'

    try:
        t240, t1440 = float(t240), float(t1440)
    except (TypeError, ValueError):
        return None, np.nan, False, 'non_numeric'

    # --- Extract confidence ---
    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except (TypeError, ValueError):
        conf = np.nan

    pred = {'target_240': t240, 'target_1440': t1440}

    # --- Optional range check ---
    if valid_range is not None:
        lo, hi = valid_range
        if not (lo <= t240 <= hi) or not (lo <= t1440 <= hi):
            return pred, conf, False, 'out_of_range'

    return pred, conf, True, None


print("All core functions defined ✓")

All core functions defined ✓


In [5]:
# ── Configuration ──────────────────────────────────────────────────
FEW_SHOT_STYLES = ["verbose", "compact"]
FEW_SHOT_COUNTS = [5, 10]

# Pre-generate few-shot example sets for each (style, count) combination
few_shot_sets = {}
for style in FEW_SHOT_STYLES:
    for n in FEW_SHOT_COUNTS:
        tag = f"{style}_{n}"
        fs = generate_few_shot(df_train, num_examples=n, features=FEATURES_FULL, style=style)
        few_shot_sets[tag] = fs
        print(f"  {tag}: {n} examples, style={style}")

print(f"\nGenerated {len(few_shot_sets)} few-shot example sets")

# Also save the few-shot example sets themselves (reusable without re-sampling)
for tag, fs in few_shot_sets.items():
    out_path = PROMPT_DIR / f"examples_{tag}.json"
    save_prompts(fs, out_path)

  verbose_5: 5 examples, style=verbose
  verbose_10: 10 examples, style=verbose
  compact_5: 5 examples, style=compact
  compact_10: 10 examples, style=compact

Generated 4 few-shot example sets
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_verbose_5.json  (3.3 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_verbose_10.json  (6.0 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_compact_5.json  (2.3 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_compact_10.json  (4.0 KB)


## Step 4: Evaluate prompt versions on a small subset

What this does: runs each prompt version on the same subset and computes parse success and accuracy.

Why it matters: small-scale testing is faster and helps pick a strong prompt before full benchmarking.

Principle: iterate quickly on a representative subset, then scale to the full test set once stable.

WARNING: Nested for loops + LLM queries = death by boredom and kernel crashes from time outs. In your actual testing keep subsets small and/or query one model at a time.

In [31]:
# Which style × feature combos to test
VERSIONS = [
    ('verbose', 'zeroshot'),
    ('verbose', '5'),
    ('verbose', '10'),
    ('compact', 'zeroshot'),
    ('compact', '5'),
    ('compact', '10'),
]

N_TEST_SAMPLES = 5
VALID_RANGE = (-10, 10)      # flag predictions outside this range

clients = {
    cfg['label']: OpenAI(
        api_key=API_KEY,
        base_url=f"http://{cfg['host']}:{cfg['port']}/v1"
    )
    for cfg in MODEL_ENDPOINTS
}

def query_llm(prompt_dict, endpoint_cfg, temperature=0.0, seed=0):
    """Send system + user prompt to an OpenAI-compatible endpoint."""
    client = clients[endpoint_cfg['label']]
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': prompt_dict['system']},
            {'role': 'user',   'content': prompt_dict['user']},
        ],
        temperature=temperature,
        seed=seed,
    )
    return response.choices[0].message.content

def build_prompt(row,style,fewshot=None):
    if fewshot is not None:
        prompt_dict = build_few_shot_prompt(fewshot, row)
    else:
        prompt_dict = build_zero_shot_prompt(row, style=style)
    return prompt_dict

def pred_llm(input_row, endpoint, version):
    style = version.split('_')[0]
    rows = version.split('_')[1]
    flag_fewshots = True
    if rows == 'zeroshot':
        few_shot = None
    else:
        fs_path = PROMPT_DIR / f'examples_{version}.json'
        if not fs_path.exists():
            raise ValueError(f"Error processing {version}: {fs_path} not found")
        few_shot = json.loads(fs_path.read_text(encoding="utf-8"))   
    prompt_dict = build_prompt(input_row,style=style,fewshot=few_shot)
    raw = query_llm(prompt_dict, endpoint, temperature=0.0, seed=0)
    pred, conf, parse_ok, parse_error = parse_response(raw, valid_range=VALID_RANGE)

    record = {
        'timestamp':      input_row[ID_COL],
        'model_label':    endpoint_cfg['label'],
        'model_name':     endpoint_cfg['model_name'],
        'endpoint_port':  endpoint_cfg['port'],
        'version':        version,
        'parse_ok':       parse_ok,
        'parse_error':    parse_error,
        'confidence':     conf,
        'raw_response':   raw,
    }
    # Ground truth
    for t in TARGET_COLS:
        record[f'gt_{t}'] = input_row[t]
    # ML predictions
    for t in ML_COLS:
        record[f'ml_{t}'] = input_row[t]
    # Predictions
    if pred is not None:
        for t in TARGET_COLS:
            record[f'pred_{t}'] = pred[t]
            record[f'err2_{t}'] = (pred[t] - input_row[t])**2
    else:
        for t in TARGET_COLS:
            record[f'pred_{t}'] = np.nan
            record[f'err2_{t}'] = np.nan
    return(record)

In [18]:
# Sample test rows (evenly spaced, no NaN)
test_subset = sample_evenly_spaced(df_test, N_TEST_SAMPLES)

records = []
for style, rows in VERSIONS:
    tag = f"{style}_{rows}" 
    for endpoint_cfg in MODEL_ENDPOINTS:
        for idx, row in test_subset.iterrows():
            record = pred_llm(row, endpoint_cfg, tag)
            records.append(record)
            print(f"  {tag} | {endpoint_cfg['label']} | ts={idx} | ok={record['parse_ok']}")

results = pd.DataFrame(records)

summary = results.groupby(
    ['model_label', 'model_name', 'endpoint_port', 'version'],
    as_index=False,
).agg(
    parse_success_rate=('parse_ok', 'mean'),
    mse_240=('err2_target_240', 'mean'),
    mse_1440=('err2_target_1440', 'mean'),
    avg_confidence=('confidence', 'mean'),
    rows=('parse_ok', 'size'),
)
summary = summary.sort_values(
    ['model_label', 'parse_success_rate', 'mse_240'],
    ascending=[True, False, True],
)

# Save
OUT_DIR = ROOT / 'Results'
results.to_csv(OUT_DIR / 'llm_inference_results.csv', index=False)
summary.to_csv(OUT_DIR / 'llm_inference_summary.csv', index=False)
display(summary)

  verbose_zeroshot | qwen2.5-7b | ts=72001 | ok=True
  verbose_zeroshot | qwen2.5-7b | ts=76590 | ok=True
  verbose_zeroshot | qwen2.5-7b | ts=84118 | ok=True
  verbose_zeroshot | qwen2.5-7b | ts=88707 | ok=True
  verbose_zeroshot | qwen2.5-7b | ts=96235 | ok=True
  verbose_zeroshot | qwen2.5-7b-instruct | ts=72001 | ok=True
  verbose_zeroshot | qwen2.5-7b-instruct | ts=76590 | ok=True
  verbose_zeroshot | qwen2.5-7b-instruct | ts=84118 | ok=True
  verbose_zeroshot | qwen2.5-7b-instruct | ts=88707 | ok=True
  verbose_zeroshot | qwen2.5-7b-instruct | ts=96235 | ok=True
  verbose_zeroshot | qwen2.5-coder-7b-instruct | ts=72001 | ok=True
  verbose_zeroshot | qwen2.5-coder-7b-instruct | ts=76590 | ok=True
  verbose_zeroshot | qwen2.5-coder-7b-instruct | ts=84118 | ok=True
  verbose_zeroshot | qwen2.5-coder-7b-instruct | ts=88707 | ok=True
  verbose_zeroshot | qwen2.5-coder-7b-instruct | ts=96235 | ok=True
  verbose_5 | qwen2.5-7b | ts=72001 | ok=False
  verbose_5 | qwen2.5-7b | ts=76590 | 

,model_label,model_name,endpoint_port,version,parse_success_rate,mse_240,mse_1440,avg_confidence,rows
5,qwen2.5-7b,Qwen2.5-7B,8000,verbose_zeroshot,1.0,1.789811e-04,3.018114e-04,0.95,5
0,qwen2.5-7b,Qwen2.5-7B,8000,compact_10,0.0,1.796280e+17,5.183341e+09,1.49,5
1,qwen2.5-7b,Qwen2.5-7B,8000,compact_5,0.0,2.081895e+17,6.708698e+09,1.49,5
2,qwen2.5-7b,Qwen2.5-7B,8000,compact_zeroshot,0.0,2.123849e+17,6.866529e+09,1.49,5
3,qwen2.5-7b,Qwen2.5-7B,8000,verbose_10,0.0,NaN,NaN,NaN,5
4,qwen2.5-7b,Qwen2.5-7B,8000,verbose_5,0.0,NaN,NaN,NaN,5
9,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,verbose_10,1.0,7.797234e-05,5.657248e-04,1.00,5
6,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,compact_10,1.0,2.321013e-04,1.701805e-03,1.00,5
7,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,compact_5,1.0,2.684723e-04,1.972687e-03,1.00,5
11,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,verbose_zeroshot,1.0,4.303109e-04,7.690392e-03,0.89,5


## Step 5: Inspect failures

What this does: shows failed parses with raw responses and error labels.

Why it matters: this evidence is what you use for the required Week 4 error analysis.

Principle: error analysis should be example-driven, not only metric-driven.

In [20]:
failures = results[~results['parse_ok']].copy()
display(failures[['model_label','version', 'raw_response', 'parse_error']])
print(f'Total failures: {len(failures)} out of {len(results)}')

,model_label,version,raw_response,parse_error
15,qwen2.5-7b,verbose_5,Q: Given features: cumulative_heat_input=42382...,invalid_json
16,qwen2.5-7b,verbose_5,Q: Given features: cumulative_heat_input=36961...,invalid_json
17,qwen2.5-7b,verbose_5,Q: Given features: cumulative_heat_input=36961...,invalid_json
18,qwen2.5-7b,verbose_5,Q: Given features: cumulative_heat_input=36961...,invalid_json
19,qwen2.5-7b,verbose_5,Q: Given features: cumulative_heat_input=50227...,invalid_json
30,qwen2.5-7b,verbose_10,Q: Given features: cumulative_heat_input=42382...,invalid_json
31,qwen2.5-7b,verbose_10,Q: Given features: cumulative_heat_input=44087...,invalid_json
32,qwen2.5-7b,verbose_10,Q: Given features: cumulative_heat_input=50255...,invalid_json
33,qwen2.5-7b,verbose_10,Q: Given features: cumulative_heat_input=54504...,invalid_json
34,qwen2.5-7b,verbose_10,Q: Given features: cumulative_heat_input=50227...,invalid_json


Total failures: 25 out of 90


## Step 6: Save final prompt files

What this does: writes your selected prompt template and few-shot examples to the submission folder.

Why it matters: prompt artifacts make your workflow auditable and reproducible for judges.

Principle: version your prompts like code. Prompt files are part of the experiment definition.

In [21]:
best_by_model = (
    summary.sort_values(['model_label', 'parse_success_rate', 'mse_240'], ascending=[True, False, True])
    .groupby('model_label', as_index=False)
    .first()
    [['model_label', 'model_name', 'endpoint_port', 'version']]
 )
display(best_by_model)

# Save one canonical template + few-shot file for submission compatibility.
overall_best_version = summary.sort_values(['parse_success_rate', 'mse_240'], ascending=[False,True]).iloc[0]['version']
print(f'Overall best prompt version: {overall_best_version}')

style = overall_best_version.split('_')[0]
fewshot_flag =  'zeroshot' not in overall_best_version
demo_row = sample_evenly_spaced(df_test, 1, FEATURES_FULL).iloc[0]
final_prompt_example = build_prompt(
    row = demo_row,
    style = style,
    fewshot = json.loads((PROMPT_DIR / f'examples_{overall_best_version}.json').read_text(encoding="utf-8")) if fewshot_flag else None
)
prompt_template_path = PROMPT_DIR / 'prompt_template.txt'
save_prompts(final_prompt_example,prompt_template_path)

# Also save model-specific prompt choices.
for _, row in best_by_model.iterrows():
    style = row.version.split('_')[0]
    fewshot_flag =  'zeroshot' not in row.version
    model_prompt = build_prompt(
        row = demo_row,
        style = style,
        fewshot = json.loads((PROMPT_DIR / f'examples_{row.version}.json').read_text(encoding="utf-8")) if fewshot_flag else None
)
    model_prompt_path = PROMPT_DIR / f"prompt_template_{row['model_label']}.txt"
    save_prompts(model_prompt,model_prompt_path)
#     print(f"Saved: {model_prompt_path}")


# print(f'Saved: {prompt_template_path}')

,model_label,model_name,endpoint_port,version
0,qwen2.5-7b,Qwen2.5-7B,8000,verbose_zeroshot
1,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,verbose_10
2,qwen2.5-coder-7b-instruct,Qwen2.5-Coder-7B-Instruct,8002,compact_10


Overall best prompt version: verbose_10
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/prompt_template.txt  (5.7 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/prompt_template_qwen2.5-7b.txt  (0.8 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/prompt_template_qwen2.5-7b-instruct.txt  (5.7 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/prompt_template_qwen2.5-coder-7b-instruct.txt  (2.6 KB)


## Step 7: Full Benchmark

What this does: loops over all configured model endpoints and a larger subset of test rows, storing raw responses and parsed predictions.

Why it matters: this produces side-by-side model outputs from a single consistent pipeline.

Principle: hold the dataset constant while varying models with prompting strategy to compare best performing versions.

In [33]:
N_TEST_LARGE = 40
test_subset_large = sample_evenly_spaced(df_test, N_TEST_LARGE)
display(test_subset_large.head())

def find_by_label(label: str) -> dict | None:
    return next((ep for ep in MODEL_ENDPOINTS if ep['label'] == label), None)

def calculate_mse(actual, predicted):
    mse = ((actual - predicted) ** 2).mean()
    return mse

def calculate_r2(actual, predicted):
    ss_res = ((actual - predicted) ** 2).sum()        # Residual Sum of Squares
    ss_tot = ((actual - actual.mean()) ** 2).sum()    # Total Sum of Squares    
    r2 = 1 - (ss_res / ss_tot)
    return r2

def evaluate(df_results,label):
    results = []
    for _, h in enumerate(HORIZONS):
        actual = df_results[f'gt_target_{h}']
        ml = df_results[f'ml_Multi-XGB_+{h}m']
        llm = df_results[f'pred_target_{h}']
        mse_llm = calculate_mse(actual, llm)
        r2_llm = calculate_r2(actual, llm)
        mse_ml = calculate_mse(actual, ml)
        r2_ml = calculate_r2(actual, ml)
        print(f"[{label}] Horizon +{h}m -> MSE_LLM: {mse_llm:.4f} | MSE_ML: {mse_ml:.4f} | R2_LLM: {r2_llm:.4f} | R2_ML: {r2_ml:.4f}")
        results.append({'Model': label, 'Horizon': h, 
                        'MSE_LLM': mse_llm, 'R2_LLM': r2_llm, 
                        'MSE_ML': mse_ml, 'R2_ML': r2_ml})
    return results

performance_metrics = []
for _,row in best_by_model.iterrows():
#     print(row)
    endpoint_config = find_by_label(row.model_label)
    records = []
    for i,rec in test_subset_large.iterrows():
        record = pred_llm(rec, endpoint_config, row.version)
        records.append(record)
    results = pd.DataFrame(records)
    results.to_csv(OUT_DIR/f"llm_benchmark_{row['model_label']}.csv")
    performance = evaluate(results,row.model_label)
    performance_metrics += performance

df_performance_metrics = pd.DataFrame(performance_metrics)
df_performance_metrics.to_csv(OUT_DIR/f"llm_benchmark_performance.csv")
display(df_performance_metrics)
    
    

,Multi-XGB_+1440m,net_flow_rolling_6h,days_since_injection,TC_INT_delta,cumulative_heat_input,target_1440,hour_sin,target_lag_60,flow_lag_15,Multi-XGB_+240m,hour_cos,T_gradient_INT_TC,target_lag_15,target_240,delta_T_above_T0_TN,timestamp,flow_lag_60,elapsed_injection_min,flow_lag_30,target_lag_30
72001,5.635816,1.485943,50.000694,0.768405,4.238254e+08,5.576771,-0.863836,5.584420,1.485913,5.606494,0.503774,-5.345621,5.570220,5.606484,5.607457,2025-02-01 20:01:00,1.485944,72001.0,1.485947,5.606272
72471,5.644828,1.485166,50.327083,0.581839,4.255806e+08,5.603088,0.845728,5.604121,1.485223,5.555615,0.533615,-5.326803,5.591828,5.580467,5.573009,2025-02-02 03:51:00,1.485338,72471.0,1.485264,5.607313
72942,5.610872,1.485213,50.654167,0.210777,4.273396e+08,5.586716,0.078459,5.586849,1.485183,5.612411,-0.996917,-5.349742,5.584277,5.555953,5.568697,2025-02-02 11:42:00,1.484978,72942.0,1.485120,5.579817
73413,5.624266,1.484812,50.981250,0.955208,4.290991e+08,5.613584,-0.918791,5.555027,1.484813,5.594944,0.394744,-5.338695,5.567058,5.569876,5.578696,2025-02-02 19:33:00,1.484759,73413.0,1.484765,5.560574
73883,5.623108,1.485418,51.307639,0.396330,4.308562e+08,5.594401,0.774393,5.588233,1.485386,5.594314,0.632705,-5.365507,5.590024,5.555054,5.607389,2025-02-03 03:23:00,1.485267,73883.0,1.485237,5.562063


[qwen2.5-7b] Horizon +240m -> MSE_LLM: 0.0003 | MSE_ML: 0.0066 | R2_LLM: 0.9780 | R2_ML: 0.5350
[qwen2.5-7b] Horizon +1440m -> MSE_LLM: 0.0009 | MSE_ML: 0.0044 | R2_LLM: 0.9338 | R2_ML: 0.6630
[qwen2.5-7b-instruct] Horizon +240m -> MSE_LLM: 0.0006 | MSE_ML: 0.0066 | R2_LLM: 0.9545 | R2_ML: 0.5350
[qwen2.5-7b-instruct] Horizon +1440m -> MSE_LLM: 0.0112 | MSE_ML: 0.0044 | R2_LLM: 0.1365 | R2_ML: 0.6630
[qwen2.5-coder-7b-instruct] Horizon +240m -> MSE_LLM: 0.0002 | MSE_ML: 0.0066 | R2_LLM: 0.9825 | R2_ML: 0.5350
[qwen2.5-coder-7b-instruct] Horizon +1440m -> MSE_LLM: 0.0026 | MSE_ML: 0.0044 | R2_LLM: 0.7962 | R2_ML: 0.6630


,Model,Horizon,MSE_LLM,R2_LLM,MSE_ML,R2_ML
0,qwen2.5-7b,240,0.000310,0.978026,0.006564,0.534982
1,qwen2.5-7b,1440,0.000858,0.933772,0.004365,0.663036
2,qwen2.5-7b-instruct,240,0.000642,0.954488,0.006564,0.534982
3,qwen2.5-7b-instruct,1440,0.011185,0.136477,0.004365,0.663036
4,qwen2.5-coder-7b-instruct,240,0.000246,0.982544,0.006564,0.534982
5,qwen2.5-coder-7b-instruct,1440,0.002639,0.796233,0.004365,0.663036


## Step 8: Draft report text

What this does: creates starter language for approach and error-analysis sections.

Why it matters: structured drafting helps teams report results consistently and quickly.

Principle: automate repetitive reporting, then refine with concrete findings from your run.

In [34]:
top_rows = summary.sort_values(['parse_success_rate', 'mse_240'], ascending=[False,True]).head(3)
display(top_rows)

approach_report = (
    'We evaluated three LLM endpoints and compared prompt versions on the same Week 3 held-out split subset. '
    'For each model, we selected a prompt based on parse reliability and accuracy. '
    'Outputs were constrained to strict JSON and validated against allowed labels before scoring.'
)

error_analysis = (
    'Common failures included malformed JSON, occasional labels outside the allowed set, and inconsistent formatting across models. '
    'Prompt wording affected compliance and quality differently by model. '
    'Few-shot prompting often improved reliability but did not remove all parsing errors.'
)

print('Approach Report draft:\n')
print(approach_report)
print('\nError Analysis draft:\n')
print(error_analysis)

,model_label,model_name,endpoint_port,version,parse_success_rate,mse_240,mse_1440,avg_confidence,rows
9,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,verbose_10,1.0,0.000078,0.000566,1.00,5
5,qwen2.5-7b,Qwen2.5-7B,8000,verbose_zeroshot,1.0,0.000179,0.000302,0.95,5
12,qwen2.5-coder-7b-instruct,Qwen2.5-Coder-7B-Instruct,8002,compact_10,1.0,0.000220,0.000675,1.00,5


Approach Report draft:

We evaluated three LLM endpoints and compared prompt versions on the same Week 3 held-out split subset. For each model, we selected a prompt based on parse reliability and accuracy. Outputs were constrained to strict JSON and validated against allowed labels before scoring.

Error Analysis draft:

Common failures included malformed JSON, occasional labels outside the allowed set, and inconsistent formatting across models. Prompt wording affected compliance and quality differently by model. Few-shot prompting often improved reliability but did not remove all parsing errors.


### TRY THIS

- Add a third prompt version
- Test a different subset size
- Add a stricter parser rule and compare failure counts
